# Shelter Infrastructure Extraction

## Objective

This notebook extracts all potential temporary shelter infrastructure from the
OpenStreetMap (OSM) regional datasets.

Unlike the previous infrastructure extraction notebook, this notebook preserves
the original building geometries and extracts additional metadata required for
estimating temporary accommodation capacity during disaster response.

The extracted shelter infrastructure will later be used to:

- Estimate available shelter capacity
- Allocate affected population to nearby shelters
- Support transportation planning
- Improve realism of disaster simulation

---

## Candidate Shelter Types

The following OSM features are considered potential temporary shelters.

### Educational Buildings

- Schools
- Colleges
- Universities

### Community Buildings

- Community Centres
- Town Halls
- Village Halls

### Sports Facilities

- Stadiums
- Sports Centres

### Emergency Shelters

- Shelter
- Refuge

### Public Buildings

- Public Buildings
- Civic Buildings

---

## Attributes Extracted

For every shelter candidate, the notebook extracts:

- Name
- Shelter Type
- Geometry
- Building Geometry Type
- Building Levels
- Capacity (if available)
- Operator
- Region Source

These attributes are later used to estimate realistic shelter capacities.

---

## Capacity Estimation Strategy

The shelter capacity will be estimated using the following priority:

1. OSM Capacity Tag (if available)

↓

2. Building Area × Number of Floors

↓

3. Building Area Only

↓

4. Default Capacity Based on Building Type

This approach produces more realistic shelter capacities than assigning fixed
values to all buildings.

---

## Input

datasets/osm_regions/

├── central-zone-260710.osm.pbf

├── eastern-zone-260710.osm.pbf

├── north-eastern-zone-260710.osm.pbf

├── northern-zone-260710.osm.pbf

├── southern-zone-260710.osm.pbf

└── western-zone-260710.osm.pbf

---

## Output

datasets/shelter_infrastructure.parquet

---

## Workflow

Regional OSM Files

↓

Extract Shelter Buildings

↓

Preserve Polygon Geometry

↓

Extract Building Attributes

↓

Estimate Building Area

↓

Estimate Shelter Capacity

↓

Merge All Regions

↓

Shelter Infrastructure Dataset

In [3]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import gc

import warnings

import numpy as np

import pandas as pd

import geopandas as gpd

from tqdm.auto import tqdm

from shapely.geometry import Point

from pyrosm import OSM

warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path.cwd().parents[1]

DATA_DIR = PROJECT_DIR / "datasets"

OSM_DIR = DATA_DIR / "osm_regions"

OUTPUT_FILE = DATA_DIR / "shelter_infrastructure.parquet"

CHECKPOINT_DIR = DATA_DIR / "osm_checkpoints"

CHECKPOINT_DIR.mkdir(

    parents=True,

    exist_ok=True

)

print("Project Directory")

print(PROJECT_DIR)

print()

print("OSM Directory")

print(OSM_DIR)

print()

print("Output File")

print(OUTPUT_FILE)

Project Directory
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination

OSM Directory
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\osm_regions

Output File
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\shelter_infrastructure.parquet


In [5]:
# ============================================================
# Candidate Shelter Tags
# ============================================================

SHELTER_AMENITIES = [

    "school",

    "college",

    "university",

    "community_centre",

    "townhall",

    "village_hall",

    "shelter",

    "social_facility"

]

SHELTER_LEISURE = [

    "stadium",

    "sports_centre"

]

SHELTER_BUILDINGS = [

    "public",

    "civic"

]

OSM_FILES = sorted(

    OSM_DIR.glob("*.osm.pbf")

)

print()

print("Regional OSM Files")

for file in OSM_FILES:

    print(file.name)

print()

print("Total Files :", len(OSM_FILES))


Regional OSM Files
central-zone-260710.osm.pbf
eastern-zone-260710.osm.pbf
north-eastern-zone-260710.osm.pbf
northern-zone-260710.osm.pbf
southern-zone-260710.osm.pbf
western-zone-260710.osm.pbf

Total Files : 6


In [6]:
# ============================================================
# Helper Functions
# ============================================================

def safe_column(gdf, column_name):

    """
    Return column if available.

    Otherwise return NaN.
    """

    if column_name in gdf.columns:

        return gdf[column_name]

    return np.nan


def add_missing_columns(gdf):

    """
    Ensure every extracted dataset
    contains the same columns.
    """

    required_columns = [

        "name",

        "amenity",

        "building",

        "leisure",

        "operator",

        "capacity",

        "building:levels",

        "geometry"

    ]

    for column in required_columns:

        if column not in gdf.columns:

            gdf[column] = np.nan

    return gdf


def determine_shelter_type(row):

    """
    Determine shelter category.
    """

    if pd.notna(row["amenity"]):

        return row["amenity"]

    if pd.notna(row["leisure"]):

        return row["leisure"]

    if pd.notna(row["building"]):

        return row["building"]

    return "Unknown"

In [7]:
# ============================================================
# Area Estimation
# ============================================================

def calculate_building_area(gdf):

    """
    Calculate polygon area in square meters.
    """

    projected = gdf.to_crs("EPSG:3857")

    area = projected.geometry.area

    return area


# ============================================================
# Capacity Estimation
# ============================================================

DEFAULT_CAPACITY = {

    "school": 600,

    "college": 1200,

    "university": 4000,

    "community_centre": 500,

    "townhall": 400,

    "village_hall": 250,

    "stadium": 10000,

    "sports_centre": 2000,

    "shelter": 300,

    "social_facility": 250,

    "public": 700,

    "civic": 1000,

    "Unknown": 300

}


def estimate_capacity(row):

    """
    Estimate shelter capacity.
    """

    # ----------------------------------------------------
    # 1. OSM Capacity
    # ----------------------------------------------------

    capacity = row.get("capacity", np.nan)

    if pd.notna(capacity):

        try:

            capacity = float(capacity)

            if np.isfinite(capacity) and capacity > 0:

                return int(capacity)

        except Exception:

            pass

    # ----------------------------------------------------
    # 2. Area
    # ----------------------------------------------------

    area = row.get("AreaSqMeters", np.nan)

    if pd.isna(area):

        area = 0

    if not np.isfinite(area):

        area = 0

    # ----------------------------------------------------
    # 3. Building Levels
    # ----------------------------------------------------

    levels = row.get("building:levels", 1)

    try:

        levels = float(levels)

    except Exception:

        levels = 1

    if not np.isfinite(levels):

        levels = 1

    if levels <= 0:

        levels = 1

    # ----------------------------------------------------
    # 4. Area Based Estimate
    # ----------------------------------------------------

    if area > 0:

        usable_area = area * levels

        estimated = usable_area / 2

        if np.isfinite(estimated):

            return max(int(estimated), 50)

    # ----------------------------------------------------
    # 5. Default
    # ----------------------------------------------------

    return DEFAULT_CAPACITY.get(

        row["ShelterType"],

        300

    )

In [8]:
# ============================================================
# Region Processing Function
# ============================================================

def process_region(osm_file):

    """
    Extract all potential shelter infrastructure
    from one regional OSM file.
    """

    print()

    print("=" * 70)

    print(osm_file.name)

    print("=" * 70)

    try:

        osm = OSM(osm_file)

        # ----------------------------------------------------
        # Combined OSM Filter
        # ----------------------------------------------------

        custom_filter = {

            "amenity": SHELTER_AMENITIES,

            "leisure": SHELTER_LEISURE,

            "building": SHELTER_BUILDINGS

        }

        shelters = osm.get_pois(

            custom_filter=custom_filter

        )

        if shelters is None or len(shelters) == 0:

            print("No Shelter Infrastructure Found")

            return None

        shelters = add_missing_columns(

            shelters

        )

        shelters["ShelterType"] = shelters.apply(

            determine_shelter_type,

            axis=1

        )

        shelters["RegionFile"] = osm_file.name

        # ----------------------------------------------------
        # Preserve Original Geometry
        # ----------------------------------------------------

        shelters["OriginalGeometry"] = (

            shelters.geometry.copy()

        )

        # ----------------------------------------------------
        # Geometry Type
        # ----------------------------------------------------

        shelters["GeometryType"] = (

            shelters.geometry.geom_type

        )

        # ----------------------------------------------------
        # Polygon Area
        # ----------------------------------------------------

        shelters["AreaSqMeters"] = np.nan

        polygon_mask = (

            shelters.geometry.geom_type

            .isin(

                [

                    "Polygon",

                    "MultiPolygon"

                ]

            )

        )

        if polygon_mask.any():

            shelters.loc[

                polygon_mask,

                "AreaSqMeters"

            ] = calculate_building_area(

                shelters.loc[polygon_mask]

            )

        # ----------------------------------------------------
        # Centroid
        # Used later for KDTree
        # ----------------------------------------------------

        centroid_geometry = (

            shelters.geometry.centroid

        )

        shelters["Latitude"] = (

            centroid_geometry.y

        )

        shelters["Longitude"] = (

            centroid_geometry.x

        )

        shelters["Centroid"] = (

            centroid_geometry

        )

        # ----------------------------------------------------
        # Estimated Capacity
        # ----------------------------------------------------

        shelters["EstimatedCapacity"] = (

            shelters.apply(

                estimate_capacity,

                axis=1

            )

        )

        print()

        print(

            "Shelter Buildings Extracted :",

            len(shelters)

        )

        print()

        print(

            shelters["ShelterType"]

            .value_counts()

        )

        return shelters

    except Exception as e:

        print()

        print("Extraction Failed")

        print(e)

        return None

    finally:

        gc.collect()

In [9]:
# ============================================================
# Process First Five Regions
# ============================================================

FIRST_FIVE = [

    file

    for file in OSM_FILES

    if "southern" not in file.name.lower()

]

all_regions = []

print()

print("=" * 80)

print("Processing First Five Regions")

print("=" * 80)

for osm_file in tqdm(

    FIRST_FIVE,

    desc="Processing"

):

    shelters = process_region(

        osm_file

    )

    if shelters is not None:

        all_regions.append(

            shelters

        )

print()

print("=" * 80)

print("First Five Regions Completed")

print("=" * 80)


Processing First Five Regions


Processing:   0%|          | 0/5 [00:00<?, ?it/s]


central-zone-260710.osm.pbf

Shelter Buildings Extracted : 4769

ShelterType
school              2695
college              901
university           297
shelter              229
stadium              185
sports_centre        145
social_facility      136
community_centre     107
townhall              38
public                31
civic                  5
Name: count, dtype: int64

eastern-zone-260710.osm.pbf

Shelter Buildings Extracted : 3625

ShelterType
school              2085
college              584
shelter              192
stadium              180
university           179
community_centre     150
sports_centre         91
townhall              72
social_facility       54
public                29
civic                  3
courthouse             2
bank                   1
post_office            1
library                1
bus_station            1
Name: count, dtype: int64

north-eastern-zone-260710.osm.pbf

Shelter Buildings Extracted : 1362

ShelterType
school              740
college  

In [10]:
# ============================================================
# Merge First Five Regions
# ============================================================

first_five = gpd.GeoDataFrame(

    pd.concat(

        all_regions,

        ignore_index=True

    ),

    geometry="geometry",

    crs="EPSG:4326"

)

checkpoint = (

    CHECKPOINT_DIR

    / "shelter_first_five.parquet"

)

first_five.to_parquet(

    checkpoint,

    index=False

)

print()

print("Checkpoint Saved")

print(checkpoint)

print()

print(first_five.shape)



Checkpoint Saved
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\osm_checkpoints\shelter_first_five.parquet

(23603, 61)


In [24]:
first_five.head()

,timestamp,lat,id,lon,changeset,visible,tags,version,addr:city,addr:country,...,AreaSqMeters,Latitude,Longitude,Centroid,EstimatedCapacity,outdoor_seating,kindergarten,building:fireproof,atm,building:min_level
0,1211604392,26.660773,266584756,79.173105,0.0,False,"{""created_by"":""Potlatch 0.9a""}",2,None,None,...,NaN,26.660773,79.173105,POINT (79.1731 26.66077),1200,None,None,None,None,None
1,1422177524,30.423991,342115702,78.085337,0.0,False,None,3,None,None,...,NaN,30.423991,78.085337,POINT (78.08534 30.42399),600,None,None,None,None,None
2,1242272740,26.969416,397129023,78.956230,0.0,False,None,1,None,None,...,NaN,26.969416,78.956230,POINT (78.95623 26.96942),10000,None,None,None,None,None
3,1255894764,26.870895,534852981,80.960557,0.0,False,None,2,None,None,...,NaN,26.870895,80.960557,POINT (80.96056 26.8709),1200,None,None,None,None,None
4,1595493857,26.852942,534853021,80.980525,0.0,False,"{""sport"":""badminton""}",2,None,None,...,NaN,26.852942,80.980525,POINT (80.98053 26.85294),2000,None,None,None,None,None


Due to memory limitations when processing the very large southern-zone-260710.osm.pbf file with Pyrosm, representative mock shelter data was generated for the Southern region. The remaining five regions use real infrastructure extracted from OpenStreetMap. The simulation pipeline remains unchanged and can be switched to real Southern OSM data in the future without modifying downstream notebooks.

In [17]:
south_mock = first_five.sample(
    n=4000,
    replace=True,
    random_state=42
).copy()

south_mock.reset_index(drop=True, inplace=True)

In [18]:
import numpy as np

np.random.seed(42)

states = [

    "Tamil Nadu",
    "Kerala",
    "Karnataka",
    "Andhra Pradesh",
    "Telangana"

]

lat_ranges = {

    "Tamil Nadu": (8.1,13.6),
    "Kerala": (8.2,12.8),
    "Karnataka": (11.5,18.5),
    "Andhra Pradesh": (13.0,19.5),
    "Telangana": (15.8,19.9)

}

lon_ranges = {

    "Tamil Nadu": (76.2,80.4),
    "Kerala": (74.8,77.5),
    "Karnataka": (74.0,78.6),
    "Andhra Pradesh": (77.0,84.8),
    "Telangana": (77.0,81.0)

}

assigned = np.random.choice(

    states,

    size=len(south_mock)

)

south_mock["State"] = assigned

south_mock["Region"] = "Southern"

In [47]:
new_lat = []

new_lon = []

for state in assigned:

    new_lat.append(

        np.random.uniform(*lat_ranges[state])

    )

    new_lon.append(

        np.random.uniform(*lon_ranges[state])

    )

south_mock["Latitude"] = new_lat
south_mock["Longitude"] = new_lon

south_mock["lat"] = new_lat
south_mock["lon"] = new_lon

In [48]:
from shapely.geometry import Point

south_mock["Centroid"] = [

    Point(lon, lat)

    for lon, lat in zip(

        south_mock["Longitude"],

        south_mock["Latitude"]

    )

]

south_mock["geometry"] = south_mock["Centroid"]

In [49]:
south_mock["id"] = np.arange(

    900000000,

    900000000 + len(south_mock)

)

In [50]:
address_columns = [

    col

    for col in south_mock.columns

    if col.startswith("addr:")

]

south_mock[address_columns] = None

In [51]:
south_mock["EstimatedCapacity"] = np.random.randint(

    200,

    1200,

    len(south_mock)

)

In [52]:
south_mock.head()

,timestamp,lat,id,lon,changeset,visible,tags,version,addr:city,addr:country,...,Longitude,Centroid,EstimatedCapacity,outdoor_seating,kindergarten,building:fireproof,atm,building:min_level,State,Region
0,1660152838,13.121475,900000000,79.783723,NaN,False,"{""fixme"":""School perimeter verification throug...",2,None,None,...,79.783723,POINT (79.78372286379762 13.121474957367946),1009,None,None,None,None,None,Andhra Pradesh,Southern
1,1602747020,19.408149,900000001,77.323408,0.0,False,None,3,None,None,...,77.323408,POINT (77.32340813215848 19.408149109858872),729,None,None,None,None,None,Telangana,Southern
2,1647200000,12.875965,900000002,78.501916,0.0,False,None,1,None,None,...,78.501916,POINT (78.50191565737012 12.875964636971451),330,None,None,None,None,None,Karnataka,Southern
3,1739723643,19.273884,900000003,80.648254,NaN,False,"{""brand"":""Kendriya Vidyalaya"",""brand:wikidata""...",2,None,None,...,80.648254,POINT (80.64825397633506 19.27388412969343),660,None,None,None,None,None,Telangana,Southern
4,1686708168,16.382833,900000004,77.845264,0.0,False,"{""contact:mobile"":""+91-83683-51277;+91-98188-7...",1,None,None,...,77.845264,POINT (77.84526383586739 16.3828331164868),468,None,None,None,None,None,Telangana,Southern


In [53]:
print("Columns only in south_mock:\n")

print(
    set(south_mock.columns) -
    set(first_five.columns)
)

print()

print("Columns only in first_five:\n")

print(
    set(first_five.columns) -
    set(south_mock.columns)
)

Columns only in south_mock:

set()

Columns only in first_five:

set()


In [54]:
# ============================================================
# Merge First Five + Mock Southern
# ============================================================

checkpoint = (

    CHECKPOINT_DIR

    / "shelter_first_five.parquet"

)

first_five = gpd.read_parquet(

    checkpoint

)
first_five["State"] = "Unknown"

first_five["Region"] = "Extracted"

shelter_gdf = gpd.GeoDataFrame(

    pd.concat(

        [

            first_five,

            south_mock

        ],

        ignore_index=True

    ),

    geometry="geometry",

    crs="EPSG:4326"

)

print()

print("Final Dataset")

print(shelter_gdf.shape)

print()

print(

    shelter_gdf["GeometryType"]

    .value_counts()

)


Final Dataset
(27603, 63)

GeometryType
Polygon            16233
Point              11321
MultiPolygon          46
MultiLineString        3
Name: count, dtype: int64


In [55]:
# ============================================================
# Data Cleaning
# ============================================================

print()

print("Rows Before Cleaning :", len(shelter_gdf))

# ------------------------------------------------------------
# Remove Missing Geometry
# ------------------------------------------------------------

shelter_gdf = shelter_gdf[

    shelter_gdf.geometry.notna()

].copy()

# ------------------------------------------------------------
# Remove Duplicate Buildings
# ------------------------------------------------------------

shelter_gdf = shelter_gdf.drop_duplicates(

    subset=[

        "name",

        "Latitude",

        "Longitude",

        "ShelterType"

    ]

)

# ------------------------------------------------------------
# Remove Invalid Geometry
# ------------------------------------------------------------

shelter_gdf = shelter_gdf[

    shelter_gdf.is_valid

].copy()

# ------------------------------------------------------------
# Capacity must be positive
# ------------------------------------------------------------

shelter_gdf = shelter_gdf[

    shelter_gdf["EstimatedCapacity"] > 0

].copy()

# ------------------------------------------------------------
# Reset Index
# ------------------------------------------------------------

shelter_gdf = shelter_gdf.reset_index(

    drop=True

)

# ------------------------------------------------------------
# Shelter ID
# ------------------------------------------------------------

shelter_gdf.insert(

    0,

    "ShelterID",

    [

        f"SH{str(i).zfill(6)}"

        for i in range(

            1,

            len(shelter_gdf)+1

        )

    ]

)

print()

print("Rows After Cleaning :", len(shelter_gdf))

print()

print("Shelter Types")

print(

    shelter_gdf["ShelterType"]

    .value_counts()

)


Rows Before Cleaning : 27603

Rows After Cleaning : 27537

Shelter Types
ShelterType
school              14682
college              4184
shelter              2734
community_centre     1237
university           1211
sports_centre        1060
stadium              1031
public                490
townhall              429
social_facility       392
civic                  33
police                  9
marketplace             9
library                 5
theatre                 5
toilets                 3
public_building         3
courthouse              3
bank                    2
post_office             2
bus_station             2
yoga                    2
Bawana Bus depot        1
music_school            1
place_of_worship        1
events_space            1
karting                 1
arts_centre             1
public_bookcase         1
hospital                1
social_centre           1
Name: count, dtype: int64


In [57]:
print(shelter_gdf.geometry.name)

print(type(shelter_gdf.geometry.iloc[0]))

print(type(shelter_gdf["Centroid"].iloc[0]))

geometry
<class 'shapely.geometry.point.Point'>
<class 'shapely.geometry.point.Point'>


In [58]:
save_gdf = shelter_gdf.drop(

    columns=["Centroid"],

    errors="ignore"

)

In [60]:
OUTPUT = DATA_DIR / "shelter_infrastructure.parquet"

save_gdf.to_parquet(

    OUTPUT,

    index=False

)

print()

print("Saved")

print(OUTPUT)


Saved
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\shelter_infrastructure.parquet
